# SCORE-Net training — Adult

This notebook trains one model and writes all artifacts to its own folder:

* `global-best-weights*.pth` — model with the best **validation** accuracy across pruning.
* `final-iter-best-weights*.pth` — final pruning iteration (highest sparsity).
* `training_history*.json` — sparsity, validation accuracy, and rule count for each iteration.
* `*-pruned.pth` — compacted copies after compression.

The test set is evaluated only once, at the end; early stopping and model selection use `val_loader`.


In [1]:
import sys
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

# Locate the project root regardless of the notebook's working directory.
cwd = Path.cwd().resolve()
project_root = next(
    (path for path in [cwd, *cwd.parents] if (path / "utils" / "experiments.py").exists()),
    cwd.parent,
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.reproducibility import set_seed
from utils.experiments import DATASETS, load_splits
from utils.model import SCORENet
from utils.train import run_iterative_pruning_pipeline
from utils.custom_utils import prune_and_compact_model

# Seed all randomness sources (Python, NumPy, PyTorch, CUDA).
SEED = set_seed(42)

if torch.cuda.is_available():
    device = "cuda"
    print(f"CUDA available! GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
elif torch.backends.mps.is_available():
    device = "mps"
    print("MPS available! GPU")
else:
    device = "cpu"
    print("CUDA not found. Running on CPU.")


CUDA available! GPU: NVIDIA GeForce GTX 1650
GPU count: 1


In [2]:
# ==============================================================================
# EXPERIMENT CONFIGURATION
# ==============================================================================
SPEC = DATASETS["adult"]
BATCH_SIZE = SPEC.batch_size
H_HIDDEN = SPEC.h_hidden
DROPOUT = SPEC.dropout
VAL_SIZE = SPEC.val_size
EXPECTED_N_FEATURES = 97

# Artifacts always go to this notebook's folder, independent of cwd.
ARTIFACT_DIR = project_root / "models/adult"
HISTORY_PATH = ARTIFACT_DIR / "training_history.json"
GLOBAL_CHECKPOINT = ARTIFACT_DIR / "global-best-weights.pth"
FINAL_CHECKPOINT = ARTIFACT_DIR / "final-iter-best-weights.pth"
GLOBAL_PRUNED = ARTIFACT_DIR / "global-best-weights-pruned.pth"
FINAL_PRUNED = ARTIFACT_DIR / "final-iter-best-weights-pruned.pth"

print(f"Artifact directory: {ARTIFACT_DIR}")
print(f"H (initial)     : {H_HIDDEN}")
print(f"Dropout         : {DROPOUT}")


Artifact directory: /home/ahmet/Projects/ScoreNetSingle/models/adult
H (initial)     : 64
Dropout         : 0.1


In [3]:
# ==============================================================================
# DATA SPLITS (TRAIN / VALIDATION / TEST)
# ==============================================================================
# load_splits: split first, stratify, and fit StandardScaler on train only.
# This keeps validation and test data out of preprocessing.
split = load_splits("adult", val_size=VAL_SIZE)

X_train_tensor, y_train_tensor = split.X_train, split.y_train
X_val_tensor, y_val_tensor = split.X_val, split.y_val
X_test_tensor, y_test_tensor = split.X_test, split.y_test

N_FEATURES = split.n_features
N_CLASSES = split.n_classes
feature_names, scaler = split.feature_names, split.scaler

if N_FEATURES != EXPECTED_N_FEATURES:
    raise ValueError(
        f"Expected feature count {EXPECTED_N_FEATURES}, data pipeline returned {N_FEATURES} "
        "returned. If preprocessing changed, update this constant and the N value in the paper."
    )

print(f"Train / validation / test: {X_train_tensor.shape[0]} / {X_val_tensor.shape[0]} / {X_test_tensor.shape[0]}")
print(f"Feature count (N)         : {N_FEATURES}")
print(f"Class count (C)           : {N_CLASSES}")
print(f"Class distribution (train): {torch.bincount(y_train_tensor).tolist()}")

# Seeded generator for reproducible batch order.
generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    split.loader("train").dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator
)
val_loader = split.loader("val", batch_size=BATCH_SIZE)
test_loader = split.loader("test", batch_size=BATCH_SIZE)


[Data] Loading UCI Adult Census Income...


-> UCI Adult ready: train shape: (32559, 97), Test Shape: (9045, 97), feature count: 97, Val Shape: (3618, 97)
Train / validation / test: 32559 / 3618 / 9045
Feature count (N)         : 97
Class count (C)           : 2
Class distribution (train): [24490, 8069]


In [4]:
# Model
# NOTE: data loading and splitting consume the global RNG. Re-seed here so
# model initialization is independent of the data path and reproducible
# (see README, "Reproducing the results").
set_seed(SEED)
model = SCORENet(N_FEATURES, H_HIDDEN, N_CLASSES, dropout_rate=DROPOUT, device=device).to(device)
print(model)


SCORENet(
  (feature_linear): Linear(in_features=97, out_features=64, bias=False)
  (output_linear): Linear(in_features=64, out_features=2, bias=False)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [5]:
# ==============================================================================
# ITERATIVE PRUNING AND RETRAINING
# ==============================================================================
# Model selection and early stopping use only val_loader.
global_best_model, final_iter_model = run_iterative_pruning_pipeline(
    model, train_loader, val_loader, device,
    prune_amount_feature=SPEC.prune_amount,
    num_prune_iters=SPEC.prune_iters,
    history_path=str(HISTORY_PATH),
)



=============== INITIAL TRAINING (BASE MODEL) STARTING ===============


Epoch [001/250] | Train Acc: 0.7893 | Val Acc: 0.8389 | LR: 0.001000


Epoch [002/250] | Train Acc: 0.8387 | Val Acc: 0.8452 | LR: 0.001000


Epoch [003/250] | Train Acc: 0.8459 | Val Acc: 0.8513 | LR: 0.001000


Epoch [004/250] | Train Acc: 0.8501 | Val Acc: 0.8543 | LR: 0.001000


Epoch [005/250] | Train Acc: 0.8522 | Val Acc: 0.8535 | LR: 0.001000


... (478 outputs left out)

Epoch [022/250] | Train Acc: 0.8531 | Val Acc: 0.8546 | LR: 0.000500


Epoch [023/250] | Train Acc: 0.8540 | Val Acc: 0.8541 | LR: 0.000500


Epoch [024/250] | Train Acc: 0.8540 | Val Acc: 0.8538 | LR: 0.000500


Epoch [025/250] | Train Acc: 0.8533 | Val Acc: 0.8510 | LR: 0.000500
[!] Early stopping triggered (20 epochs without improvement). Best in cycle: 0.8557

========================= TRAINING COMPLETE =========================
1. Model: best overall (Global Best Val Acc: 0.8618)
2. Model: 15. best of the final iteration (Final Pruned Model)

[HISTORY] Training history saved: /home/ahmet/Projects/ScoreNetSingle/models/adult/training_history.json


In [6]:
# Raw (pre-compaction) checkpoints loaded by the analysis notebooks.
torch.save(global_best_model.state_dict(), GLOBAL_CHECKPOINT)
torch.save(final_iter_model.state_dict(), FINAL_CHECKPOINT)
print(f"Saved: {GLOBAL_CHECKPOINT}")
print(f"Saved: {FINAL_CHECKPOINT}")


Saved: /home/ahmet/Projects/ScoreNetSingle/models/adult/global-best-weights.pth
Saved: /home/ahmet/Projects/ScoreNetSingle/models/adult/final-iter-best-weights.pth


In [7]:
# ==============================================================================
# COMPACTION (STRUCTURAL + EMPIRICAL DEAD-RULE CLEANUP, EXACT BIAS SHIFTING)
# ==============================================================================
global_best_model = prune_and_compact_model(global_best_model, train_loader)
final_iter_model = prune_and_compact_model(final_iter_model, train_loader)

torch.save(global_best_model.state_dict(), GLOBAL_PRUNED)
torch.save(final_iter_model.state_dict(), FINAL_PRUNED)
print(f"Saved: {GLOBAL_PRUNED}")
print(f"Saved: {FINAL_PRUNED}")



MODEL COMPACTION AND DEAD-RULE CLEANUP REPORT
Rule capacity before compaction (H): 64
-> Structurally empty rules (<= 0.0001) : 13
-> Empirically dead rules (never triggered): 0
Total removed rules: 13 (removal rate: %20.3)
Active surviving rules (new H): 51
[OK] Pruning and compaction complete.
-> Net bias contribution transferred to classes:
[0. 0.]
-> New rule count (H): 51




MODEL COMPACTION AND DEAD-RULE CLEANUP REPORT
Rule capacity before compaction (H): 64
-> Structurally empty rules (<= 0.0001) : 21
-> Empirically dead rules (never triggered): 0
Total removed rules: 21 (removal rate: %32.8)
Active surviving rules (new H): 43
[OK] Pruning and compaction complete.
-> Net bias contribution transferred to classes:
[-0.1543  0.0668]
-> New rule count (H): 43

Saved: /home/ahmet/Projects/ScoreNetSingle/models/adult/global-best-weights-pruned.pth
Saved: /home/ahmet/Projects/ScoreNetSingle/models/adult/final-iter-best-weights-pruned.pth


In [8]:
# ==============================================================================
# TEST EVALUATION (final only, once)
# ==============================================================================
global_best_model.eval()
final_iter_model.eval()

with torch.no_grad():
    for model_name, trained_model in [
        ("Final Iter (compacted)", final_iter_model),
        ("Global Best (compacted)", global_best_model),
    ]:
        preds = trained_model(X_test_tensor.to(device)).argmax(dim=1).cpu()
        acc = (preds == y_test_tensor).float().mean().item()
        print(f"{model_name:26s} | H={trained_model.H:5d} | Test Accuracy: {acc:.4f}")


Final Iter (compacted)     | H=   43 | Test Accuracy: 0.8561
Global Best (compacted)    | H=   51 | Test Accuracy: 0.8530
